# Kaiji Stage1 Offline Analysis Walkthrough

This notebook executes the local static-analysis pipeline for sample `0a70...a71` and surfaces persistence, C2, and attack-module evidence.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

_cwd = Path.cwd().resolve()
ROOT = _cwd.parent if _cwd.name == 'notebooks' else _cwd
SCRIPTS = ROOT / 'scripts'
SAMPLE = ROOT / 'input' / '0a70d7699c8e0629597dcc03b1aef0beebec03ae0580f2c070fb2bfd2fd89a71.elf'
ROOT, SAMPLE.exists()

## Run Full Pipeline

In [ ]:
subprocess.run([sys.executable, str(SCRIPTS / 'run_full_analysis.py'), '--root', str(ROOT)], check=True)
subprocess.run([sys.executable, str(SCRIPTS / 'extract_rodata_artifacts.py'), '--sample', str(SAMPLE), '--out-json', str(ROOT / 'reports/json/rodata_artifacts.json'), '--out-md', str(ROOT / 'reports/static/rodata_artifacts.md')], check=True)

## Summary Artifacts

In [ ]:
triage = json.loads((ROOT / 'reports/json/triage_report.json').read_text())
cfg = json.loads((ROOT / 'reports/json/config_report.json').read_text())
persist = json.loads((ROOT / 'reports/json/persistence_blocks.json').read_text())
matrix = json.loads((ROOT / 'reports/json/capability_matrix.json').read_text())
summary = {
    'sample_sha256': triage['sample']['sha256'],
    'size_bytes': triage['sample']['size_bytes'],
    'persistence_hits': len(triage['persistence_indicators']),
    'persistence_block_hits': persist['hit_count'],
    'behavior_hits': len(triage['behavior_indicators']),
    'decoded_tokens': cfg['decoded_config_tokens'],
    'c2_candidates': cfg['c2_candidates'],
    'ares_functions': triage['go_indicators']['ares_functions'][:8],
    'capability_buckets': matrix['summary']['capability_counts'],
}
summary


## Persistence Block Recovery\n
\n
These values come from printable-run carving around persistence markers (systemd + cron).

In [ ]:
persist['extracted']


## Capability Matrix (Observed Symbol -> Inferred Capability)

In [ ]:
matrix['summary']


## Confidence Rubric (Observed vs Inferred)

In [ ]:
evidence_rubric = [
    {'type': 'observed', 'evidence': 'Embedded Base64 token decodes to air.xem.lat:25194|(odk)/*-'},
    {'type': 'observed', 'evidence': 'systemd/cron persistence strings and service template lines present'},
    {'type': 'observed', 'evidence': 'Ares_* and Killcpu symbol strings present in .rodata/.gopclntab context'},
    {'type': 'inferred', 'evidence': 'Botnet DDoS role based on module naming and persistence behavior'},
    {'type': 'inferred', 'evidence': 'Kaiji-like family alignment, pending runtime confirmation of protocol behavior'},
]
evidence_rubric


## Quick Validation Checks\n\n- `air.xem.lat:25194` should appear in decoded config tokens.\n- Persistence paths should include `quotaoff.service` and `/etc/crontab`.\n- Persistence block extraction should produce at least one service/cron block.

In [ ]:
checks = {
    'decoded_air_xem': any('air.xem.lat:25194' in t for t in cfg['decoded_config_tokens']),
    'has_quotaoff': any('quotaoff' in p for p in cfg['persistence_paths']),
    'has_crontab': any('/etc/crontab' in p for p in cfg['persistence_paths']),
    'persist_block_count': persist['hit_count'],
}
checks
